# 2. Data Preprocessing

This notebook prepares the ISOT Fake News Dataset for machine learning based on the issues identified during exploratory data analysis.

The preprocessing strategy focuses on removing obvious source-specific and scraping artifacts while preserving meaningful linguistic information. Particular attention is given to preventing data leakage during the train-test preparation process.

## 2.1 Data Loading

In [74]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import re
import string

In [75]:
fake_df = pd.read_csv("../data/raw/Fake.csv")
true_df = pd.read_csv("../data/raw/True.csv")

In [76]:
print("Fake news:", fake_df.shape)
print("Real news:", true_df.shape)

Fake news: (23481, 4)
Real news: (21417, 4)


## 2.2 Dataset Preparation


In [77]:
fake_df["label"] = 0
true_df["label"] = 1

df = pd.concat([fake_df, true_df], ignore_index=True)

### Structural Cleaning

Based on the exploratory analysis, exact duplicate records and observations with empty article bodies are removed before text preprocessing.

In [78]:
df = df.drop_duplicates().reset_index(drop=True)

In [79]:
df = df[df["text"].str.strip() != ""].reset_index(drop=True)

In [80]:
print("Duplicate texts:", df.duplicated(subset=["text"]).sum())

label_conflicts = (
    df.groupby("text")["label"]
      .nunique()
      .gt(1)
      .sum()
)

print("Identical texts with conflicting labels:", label_conflicts)

Duplicate texts: 5414
Identical texts with conflicting labels: 0


In [81]:
df = (
    df.drop_duplicates(subset=["text"])
      .reset_index(drop=True)
)

In [82]:
print("Dataset shape after text deduplication:", df.shape)
print("Duplicate texts remaining:", df.duplicated(subset=["text"]).sum())

Dataset shape after text deduplication: (38644, 5)
Duplicate texts remaining: 0


In [83]:
print("Dataset shape:", df.shape)
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate texts:", df.duplicated(subset=["text"]).sum())
print("Empty texts:", (df["text"].str.strip() == "").sum())

Dataset shape: (38644, 5)
Duplicate rows: 0
Duplicate texts: 0
Empty texts: 0


## 2.3 Text Cleaning

The EDA revealed several source-specific and scraping artifacts that could provide shortcuts to the classifier. The text is therefore cleaned to reduce these signals while preserving meaningful linguistic content.

The cleaning process focuses on:
- removing URLs and web-related artifacts;
- removing explicit Reuters source markers;
- normalizing text formatting;
- preserving potentially meaningful lexical information.

In [84]:
def clean_text(text):
    # Remove URLs
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text,
        flags=re.IGNORECASE
    )

    # Remove explicit source marker
    text = re.sub(
        r'\(\s*reuters\s*\)\s*-?',
        ' ',
        text,
        flags=re.IGNORECASE
    )
    text = re.sub(
        r'\breuters\b',
        ' ',
        text,
        flags=re.IGNORECASE
    )



    # Remove scraping / publishing artifacts
    artifacts = [
        r'\bfeatured image\b',
        r'\bgetty images\b',
        r'\bpic twitter\b',
        r'\btwitter com\b',
        r'\bscreen capture\b'
    ]

    for pattern in artifacts:
        text = re.sub(pattern, ' ', text, flags=re.IGNORECASE)

    # Normalize case
    text = text.lower()

    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [85]:
df["clean_text"] = df["text"].apply(clean_text)

In [86]:
patterns_to_check = [
    r'https?://|www\.',
    r'\breuters\b',
    r'\bfeatured image\b',
    r'\bgetty images\b',
    r'\bpic twitter\b',
    r'\btwitter com\b',
    r'\bscreen capture\b'
]

for pattern in patterns_to_check:
    count = df["clean_text"].str.contains(
        pattern,
        case=False,
        regex=True,
        na=False
    ).sum()

    print(pattern, ":", count)

https?://|www\. : 0
\breuters\b : 0
\bfeatured image\b : 0
\bgetty images\b : 0
\bpic twitter\b : 0
\btwitter com\b : 0
\bscreen capture\b : 0


In [87]:
print("Empty clean texts:", (df["clean_text"] == "").sum())

Empty clean texts: 53


In [89]:
df = (
    df[df["clean_text"] != ""]
    .reset_index(drop=True)
)

In [90]:
df[["text", "clean_text"]].sample(5, random_state=42)

,text,clean_text
29778,MOGADISHU (Reuters) - A Somali television jour...,mogadishu a somali television journalist was k...
6252,White privilege is a thing. There are many whi...,white privilege is a thing. there are many whi...
29269,DUBAI (Reuters) - Eight women and two children...,dubai eight women and two children from the sa...
217,Everyone knows what a dirtbag Donald Trump is ...,everyone knows what a dirtbag donald trump is ...
15576,This report is so outrageous that we could har...,this report is so outrageous that we could har...


In [91]:
print("Dataset shape:", df.shape)
print("Empty clean texts:", (df["clean_text"] == "").sum())

Dataset shape: (38591, 6)
Empty clean texts: 0


### Text Cleaning Summary

The text preprocessing strategy was intentionally kept conservative in order to preserve as much linguistic information as possible.

The following transformations were applied:
- URLs were removed.
- Explicit source markers such as `Reuters` were removed to reduce source-related leakage.
- Scraping and publishing artifacts identified during the EDA were removed.
- Text was converted to lowercase.
- Whitespace was normalized.
- Records containing no usable text after preprocessing were removed.

Punctuation, numbers, and stopwords were intentionally preserved at this stage to avoid unnecessarily removing potentially useful information.

After preprocessing, the dataset contains **38,591 articles**, and all remaining records contain non-empty cleaned text.

### Train-Test Split

The cleaned dataset is divided into training and test sets using an 80/20 split.

The resulting datasets contain:
- **30,872 training samples**
- **7,719 test samples**

Both subsets preserve the overall class distribution, with approximately **54.9% real news** and **45.1% fake news**.


In [92]:
X = df["clean_text"]
y = df["label"]

In [93]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
)

##### Vocabulary Filtering

An initial TF-IDF vectorization without document-frequency filtering produced more than 106,000 features, indicating the presence of a large number of very rare terms.

Different minimum document-frequency thresholds were evaluated:

- `min_df=1`: 106,120 features
- `min_df=2`: 55,055 features
- `min_df=5`: 33,352 features
- `min_df=10`: 23,951 features

A threshold of `min_df=5` was selected for the baseline. This removes terms appearing in fewer than five training documents while retaining a sufficiently rich vocabulary for text classification.

In [94]:
print("X_train:", X_train.shape)
print("X_test :", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True))

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True))

X_train: (30872,)
X_test : (7719,)

Train class distribution:
label
1    0.549106
0    0.450894
Name: proportion, dtype: float64

Test class distribution:
label
1    0.549164
0    0.450836
Name: proportion, dtype: float64


In [95]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [99]:
for min_df in [1, 2, 5, 10]:
    vectorizer = TfidfVectorizer(min_df=min_df)
    X_temp = vectorizer.fit_transform(X_train)

    print(f"min_df={min_df}: {X_temp.shape[1]} features")

min_df=1: 106120 features
min_df=2: 55055 features
min_df=5: 33352 features
min_df=10: 23951 features


In [100]:
tfidf = TfidfVectorizer(
    min_df=5
)

In [101]:
X_train_tfidf = tfidf.fit_transform(X_train)

In [102]:
X_test_tfidf = tfidf.transform(X_test)

In [103]:
print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape:", X_test_tfidf.shape)

Train TF-IDF shape: (30872, 33352)
Test TF-IDF shape: (7719, 33352)


# Preprocessing Conclusion

The raw ISOT datasets were labelled, merged, structurally cleaned, and prepared for machine learning.

The preprocessing pipeline included:
- removal of exact and text-level duplicates;
- removal of empty article bodies;
- removal of URLs, explicit Reuters markers, and selected scraping artifacts;
- lowercase normalization and whitespace normalization;
- an 80/20 stratified train-test split;
- TF-IDF vectorization fitted exclusively on the training data.

A minimum document frequency of 5 was selected, reducing the vocabulary from 106,120 to **33,352 features**.

The final representation contains:
- **30,872 training samples × 33,352 features**
- **7,719 test samples × 33,352 features**
